In [59]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from joblib import Parallel, delayed
from scipy.stats import hmean
from matplotlib.ticker import PercentFormatter
import re

In [60]:
nifd_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/NIFD/test_cog')
nacc_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/nacc_test_updated/test_cog')
adni_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/ADNI/test_cog')
ppmi_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/PPMI/test_cog')
brainlat_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/brainlat/test_cog')

In [61]:
def option_string_to_dict(options):
    pattern = r"([A-Z])\. ([^\n]+)"
    matches = re.findall(pattern, options)
    return {key: value for key, value in matches}

def load_answers(dir_path, dataset_name):
    # load all parquet files from the directory, stack them into a pandas datafame
    # this only reads the participant ID, ground trush answer and the prediction. 
    # Reading only those columns is significantly faster (about 100x) than loading the whole dataframe.
    # Loading everything is very slow because there are extremely long strings (model outputs) in some columns

    fpaths = list(dir_path.rglob('*.parquet'))

    dfs = []

    cols_to_read = ['ID','ground_truth','prediction','ground_truth_text','options']

    for fpath in tqdm(fpaths):

        model = fpath.parent.name.split('-',3)[-1] 
        benchmark = fpath.parent.parent.name.split('_',1)[-1].upper()

        df = pd.read_parquet(fpath,columns=cols_to_read)
    
        df = df.assign(model=model, benchmark=benchmark)

        df['correct'] = (df['ground_truth'] == df['prediction']).astype(int)

        df['prediction_text'] = df.apply(lambda row: option_string_to_dict(row['options']).get(row['prediction'],'invalid'),axis=1)

        dfs.append(df)

    df = pd.concat(dfs)
    df['dataset'] = dataset_name

    # make these columns Categorical
    group_cols = ["dataset", "benchmark", "model", "ground_truth_text", 'prediction_text']
    for col in group_cols:
        df[col] = pd.Categorical(df[col])

    # keep only the models we actually care about
    df = df[df['model'].isin(['Qwen2.5-3B-Instruct','Qwen2.5-7B-Instruct','NACC-3B-OS-SCE'])]

    return df


In [62]:
nacc_res = load_answers(nacc_path,dataset_name='NACC')

100%|██████████| 9/9 [00:09<00:00,  1.03s/it]


In [63]:
brainlat_res = load_answers(brainlat_path,dataset_name='BrainLat')

100%|██████████| 8/8 [00:00<00:00, 32.46it/s]


In [64]:
ppmi_res = load_answers(ppmi_path,dataset_name='PPMI')

100%|██████████| 8/8 [00:01<00:00,  7.72it/s]


In [65]:
nifd_res = load_answers(nifd_path,dataset_name='NIFD')

100%|██████████| 8/8 [00:00<00:00, 62.96it/s]


In [66]:
adni_res = load_answers(adni_path,dataset_name='ADNI')

100%|██████████| 8/8 [00:00<00:00, 17.39it/s]


In [67]:
nifd_res.sample()

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,dataset
1271,1_S_0011,C,C,Dementia (DE),A. Normal Cognition (NC)\nB. Mild Cognitive Im...,NACC-3B-OS-SCE,COG,1,Dementia (DE),NIFD


In [ ]:
from itertools import combinations


def _process_group_with_bootstrap_samples(id, group, n_boot, seed):
    """
    Modified helper function that returns raw bootstrap samples in addition to CIs.
    This allows us to perform statistical tests on the bootstrap distributions.
    """
    dataset, model = id
    n_samples = len(group)
    
    if n_samples == 0:
        return [], {}

    unique_labels = group['ground_truth_text'].unique()
    
    # --- Reproducible RNG per worker ---
    rng = np.random.default_rng(seed)
    
    # Pre-generate all bootstrap indices at once
    bootstrap_indices = rng.integers(0, n_samples, size=(n_boot, n_samples))
    
    # Pre-allocate dicts to store results per class
    boot_results = {}
    for label in unique_labels:
        boot_results[('precision', label)] = []
        boot_results[('recall', label)] = []
    
    # Convert to NumPy arrays ONCE per group
    y_true = group["ground_truth_text"].to_numpy()
    y_pred = group["prediction_text"].to_numpy()
    
    for indices in bootstrap_indices:
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # Get per-class metrics
        p, r, f, s = precision_recall_fscore_support(
            y_true_boot,
            y_pred_boot,
            average=None,
            labels=unique_labels,
            zero_division=0,
        )
        
        # Store per-class results
        for i, label in enumerate(unique_labels):
            boot_results[('precision', label)].append(p[i])
            boot_results[('recall', label)].append(r[i])

    # --- Calculate quantiles ---
    res_list = []
    bootstrap_samples = {}  # Store raw samples for statistical tests
    
    for label in unique_labels:
        # Process precision for this class
        precision_values = np.array(boot_results[('precision', label)])
        
        low_idx = int(0.025 * n_boot)
        med_idx = int(0.5 * n_boot)
        high_idx = int(0.975 * n_boot)
        
        partitioned = np.partition(precision_values, [low_idx, med_idx, high_idx])
        
        res_list.append({
            "dataset": dataset,
            "model": model,
            "class": label,
            "metric": "precision",
            "median": partitioned[med_idx],
            "low": partitioned[low_idx],
            "high": partitioned[high_idx],
        })
        
        # Store bootstrap samples
        bootstrap_samples[(dataset, model, label, "precision")] = precision_values
        
        # Process recall for this class
        recall_values = np.array(boot_results[('recall', label)])
        partitioned = np.partition(recall_values, [low_idx, med_idx, high_idx])
        
        res_list.append({
            "dataset": dataset,
            "model": model,
            "class": label,
            "metric": "recall",
            "median": partitioned[med_idx],
            "low": partitioned[low_idx],
            "high": partitioned[high_idx],
        })
        
        # Store bootstrap samples
        bootstrap_samples[(dataset, model, label, "recall")] = recall_values
        
    return res_list, bootstrap_samples


def optimized_bootstrap_parallel_with_samples(df, n_boot, seed=None, n_jobs=-1):
    """
    Compute bootstrap confidence intervals and return raw bootstrap samples.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with columns: dataset, model, ground_truth_text, prediction_text
    n_boot : int
        Number of bootstrap samples
    seed : int, optional
        Random seed for reproducible results
    n_jobs : int, optional
        Number of CPU cores to use. -1 means all cores.
    
    Returns
    -------
    results_df : pd.DataFrame
        Results with columns: dataset, model, class, metric, median, low, high
    bootstrap_samples : dict
        Dictionary mapping (dataset, model, class, metric) -> array of bootstrap samples
    """
    
    # --- Setup ---
    main_rng = np.random.default_rng(seed)
    
    df_copy = df.copy()
    group_cols = ["dataset", "model"]
            
    # Get all groups
    groups = list(df_copy.groupby(group_cols, observed=True))
    
    # Generate a unique seed for each group
    n_groups = len(groups)
    group_seeds = main_rng.integers(0, 2**32, size=n_groups)
    
    # --- Parallel Execution ---
    results_with_samples = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_process_group_with_bootstrap_samples)(
            id, 
            group, 
            n_boot, 
            group_seeds[i]
        )
        for i, (id, group) in enumerate(groups)
    )
    
    # --- Collect Results ---
    final_results = []
    all_bootstrap_samples = {}
    
    for res_list, boot_samples in results_with_samples:
        final_results.extend(res_list)
        all_bootstrap_samples.update(boot_samples)
    
    return pd.DataFrame(final_results), all_bootstrap_samples


def permutation_test_bootstrap_medians(samples1, samples2, n_permutations=10000, seed=None):
    """
    Perform a permutation test comparing medians of two bootstrap distributions.
    
    Parameters
    ----------
    samples1 : np.ndarray
        Bootstrap samples from first model
    samples2 : np.ndarray
        Bootstrap samples from second model
    n_permutations : int
        Number of permutations to perform
    seed : int, optional
        Random seed for reproducibility
    
    Returns
    -------
    p_value : float
        Two-sided p-value
    """
    rng = np.random.default_rng(seed)
    
    # Observed difference in medians
    observed_diff = np.median(samples1) - np.median(samples2)
    
    # Combine samples
    combined = np.concatenate([samples1, samples2])
    n1 = len(samples1)
    n_total = len(combined)
    
    # Permutation test
    null_diffs = np.zeros(n_permutations)
    
    for i in range(n_permutations):
        # Shuffle combined samples
        perm_idx = rng.permutation(n_total)
        perm1 = combined[perm_idx[:n1]]
        perm2 = combined[perm_idx[n1:]]
        
        # Compute difference in medians under null
        null_diffs[i] = np.median(perm1) - np.median(perm2)
    
    # Two-sided p-value
    p_value = np.mean(np.abs(null_diffs) >= np.abs(observed_diff))
    
    return p_value


def compute_pairwise_comparisons(bootstrap_samples, n_permutations=10000, seed=None, n_jobs=-1):
    """
    Compute pairwise permutation tests between all models for each dataset/class/metric combination.
    
    Parameters
    ----------
    bootstrap_samples : dict
        Dictionary from optimized_bootstrap_parallel_with_samples
        Maps (dataset, model, class, metric) -> array of bootstrap samples
    n_permutations : int
        Number of permutations for each test
    seed : int, optional
        Random seed for reproducibility
    n_jobs : int, optional
        Number of CPU cores to use. -1 means all cores.
    
    Returns
    -------
    pd.DataFrame
        Results with columns: dataset, class, metric, model1, model2, p_value
    """
    
    # Extract unique combinations of dataset, class, metric
    keys = list(bootstrap_samples.keys())
    unique_combos = set((dataset, cls, metric) for dataset, model, cls, metric in keys)
    
    # For each combination, get all models
    comparison_tasks = []
    
    for dataset, cls, metric in unique_combos:
        # Find all models for this combination
        models = [model for d, model, c, m in keys 
                  if d == dataset and c == cls and m == metric]
        models = sorted(set(models))
        
        # Generate all pairwise comparisons
        for model1, model2 in combinations(models, 2):
            key1 = (dataset, model1, cls, metric)
            key2 = (dataset, model2, cls, metric)
            
            if key1 in bootstrap_samples and key2 in bootstrap_samples:
                comparison_tasks.append({
                    'dataset': dataset,
                    'class': cls,
                    'metric': metric,
                    'model1': model1,
                    'model2': model2,
                    'samples1': bootstrap_samples[key1],
                    'samples2': bootstrap_samples[key2]
                })
    
    # Set up RNG for reproducibility
    main_rng = np.random.default_rng(seed)
    task_seeds = main_rng.integers(0, 2**32, size=len(comparison_tasks))
    
    # Parallel execution of permutation tests
    def run_single_test(task, task_seed):
        p_value = permutation_test_bootstrap_medians(
            task['samples1'],
            task['samples2'],
            n_permutations=n_permutations,
            seed=task_seed
        )
        return {
            'dataset': task['dataset'],
            'class': task['class'],
            'metric': task['metric'],
            'model1': task['model1'],
            'model2': task['model2'],
            'p_value': p_value,
            'median1': np.median(task['samples1']),
            'median2': np.median(task['samples2']),
            'median_diff': np.median(task['samples1']) - np.median(task['samples2'])
        }
    
    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(run_single_test)(task, task_seeds[i])
        for i, task in enumerate(comparison_tasks)
    )
    
    return pd.DataFrame(results)

In [69]:
results_df = pd.concat(
    [
        adni_res,
        brainlat_res,
        nifd_res,
        nacc_res,
        ppmi_res
    ]
)

In [78]:
# Step 1: Compute bootstrap CIs with samples
all_metrics, bootstrap_samples = optimized_bootstrap_parallel_with_samples(
    results_df, n_boot=1000, seed=42, n_jobs=-1
)

In [80]:
all_metrics

,dataset,model,class,metric,median,low,high
0,ADNI,NACC-3B-OS-SCE,Normal Cognition (NC),precision,0.660758,0.644404,0.677028
1,ADNI,NACC-3B-OS-SCE,Normal Cognition (NC),recall,0.773435,0.756982,0.788846
2,ADNI,NACC-3B-OS-SCE,Mild Cognitive Impairment (MCI),precision,0.590845,0.573592,0.608319
3,ADNI,NACC-3B-OS-SCE,Mild Cognitive Impairment (MCI),recall,0.556072,0.538588,0.572851
4,ADNI,NACC-3B-OS-SCE,Dementia (DE),precision,0.685083,0.654335,0.714625
...,...,...,...,...,...,...,...
73,PPMI,Qwen2.5-7B-Instruct,Mild Cognitive Impairment (MCI),recall,0.748994,0.729834,0.765898
74,PPMI,Qwen2.5-7B-Instruct,Normal Cognition (NC),precision,0.948010,0.943821,0.952413
75,PPMI,Qwen2.5-7B-Instruct,Normal Cognition (NC),recall,0.724717,0.717418,0.732116
76,PPMI,Qwen2.5-7B-Instruct,Dementia (DE),precision,0.552036,0.489083,0.615385


In [81]:
# Step 2: Compute pairwise comparisons
pairwise_pvalues = compute_pairwise_comparisons(
    bootstrap_samples, n_permutations=1000, seed=42, n_jobs=-1
)

# The pairwise_pvalues dataframe contains:
# - dataset, class, metric: identifies the comparison context
# - model1, model2: the two models being compared
# - p_value: permutation test p-value
# - median1, median2: median values for each model
# - median_diff: difference in medians (model1 - model2)

In [94]:
pairwise_pvalues.p_value.to_numpy()

array([0.   , 0.906, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   ])

In [82]:
pairwise_pvalues.sort_values(['dataset','metric']).head(10)

,dataset,class,metric,model1,model2,p_value,median1,median2,median_diff
18,ADNI,Normal Cognition (NC),precision,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0,0.660756,0.663382,-0.002627
19,ADNI,Normal Cognition (NC),precision,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,0.0,0.660756,0.682064,-0.021309
20,ADNI,Normal Cognition (NC),precision,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,0.0,0.663382,0.682064,-0.018682
24,ADNI,Mild Cognitive Impairment (MCI),precision,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0,0.590829,0.526815,0.064014
25,ADNI,Mild Cognitive Impairment (MCI),precision,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,0.0,0.590829,0.570203,0.020627
26,ADNI,Mild Cognitive Impairment (MCI),precision,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,0.0,0.526815,0.570203,-0.043388
33,ADNI,Dementia (DE),precision,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0,0.685047,0.535375,0.149673
34,ADNI,Dementia (DE),precision,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,0.0,0.685047,0.746296,-0.061249
35,ADNI,Dementia (DE),precision,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,0.0,0.535375,0.746296,-0.210922
45,ADNI,Normal Cognition (NC),recall,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0,0.773435,0.636609,0.136826


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.patches import Rectangle

sns.set_style("whitegrid")


def get_significance_marker(p_value):
    """Convert p-value to significance marker."""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    else:
        return "ns"  # Not significant


def format_pvalue(p_value):
    """Format p-value for display (alternative version)."""
    if p_value < 0.001:
        return "p<0.001"
    else:
        return f"p={p_value:.3f}"


def plot_classwise_with_pvalues(all_metrics, pairwise_pvalues, 
                                  model_map, class_map,
                                  output_dir="../figure2_classwise",
                                  show_all_comparisons=False,
                                  p_threshold=0.05):
    """
    Plot classwise precision and recall with p-value annotations.
    
    Parameters
    ----------
    all_metrics : pd.DataFrame
        Results with columns: dataset, model, class, metric, median, low, high
    pairwise_pvalues : pd.DataFrame
        P-values with columns: dataset, class, metric, model1, model2, p_value
    model_map : dict
        Mapping from full model names to abbreviations
    class_map : dict
        Mapping from full class names to abbreviations
    output_dir : str
        Directory to save the figure
    show_all_comparisons : bool
        If True, show all pairwise comparisons. If False, only show comparisons
        involving the first model (typically baseline)
    p_threshold : float
        Only show annotations for p-values below this threshold
    """
    
    # Apply mappings
    all_metrics = all_metrics.copy()
    all_metrics["model_abbrev"] = all_metrics["model"].map(model_map)
    all_metrics["class_abbrev"] = all_metrics["class"].map(class_map).fillna(all_metrics["class"])
    
    # Apply mappings to pairwise_pvalues
    pvalues = pairwise_pvalues.copy()

    # Map abbreviated names to full names for matching
    pvalues["model1_abbrev"] = pvalues["model1"].apply(
        lambda x: model_map.get(x, x)
    )
    pvalues["model2_abbrev"] = pvalues["model2"].apply(
        lambda x: model_map.get(x, x)
    )
    pvalues["class_abbrev"] = pvalues["class"].map(class_map).fillna(pvalues["class"])
    
    # Filter for precision and recall only
    df = all_metrics[all_metrics['metric'].isin(['precision', 'recall'])].copy()
    
    # Get unique values
    datasets = sorted(df["dataset"].unique())
    models = ["Q3B", "LUNAR", "Q7B"]
    
    # Define colors for models
    palette = dict(zip(models, sns.color_palette("colorblind", n_colors=len(models))))
    
    # Create figure: rows = metrics, columns = datasets
    metrics = ['precision', 'recall']
    metric_names = {'precision': 'Precision', 'recall': 'Recall'}
    n_datasets = len(datasets)
    
    fig, axes = plt.subplots(2, n_datasets, figsize=(4 * n_datasets, 6))
    
    # Ensure axes is 2D even with one dataset
    if n_datasets == 1:
        axes = axes.reshape(-1, 1)
    
    # Fixed orders
    class_order = ["NC", "MCI", "IMP", "DE"]
    dataset_order = ["NACC", "ADNI", "BrainLat", "NIFD", "PPMI"]
    
    # Override datasets list
    datasets = [d for d in dataset_order if d in df["dataset"].unique()]
    
    # Set up consistent bar width based on maximum possible classes
    n_models = len(models)
    bar_width = 0.8 / n_models
    
    for col_idx, dataset in enumerate(datasets):
        # Get classes for this specific dataset (using abbreviated names)
        present = df[df['dataset'] == dataset]['class_abbrev'].unique()
        dataset_classes = [c for c in class_order if c in present]
        
        for row_idx, metric in enumerate(metrics):
            ax = axes[row_idx, col_idx]
            
            # Filter data for this dataset and metric
            df_subset = df[(df['dataset'] == dataset) & (df['metric'] == metric)]
            
            # Store bar positions and heights for annotation
            bar_info = {model: {'x': [], 'height': []} for model in models}
            
            # For each model, plot bars
            for i, model in enumerate(models):
                df_model = df_subset[df_subset['model_abbrev'] == model]
                
                # Get values in class order
                values = []
                errors_low = []
                errors_high = []
                
                for cls in dataset_classes:
                    cls_data = df_model[df_model['class_abbrev'] == cls]
                    if len(cls_data) > 0:
                        values.append(cls_data['median'].values[0])
                        errors_low.append(cls_data['median'].values[0] - cls_data['low'].values[0])
                        errors_high.append(cls_data['high'].values[0] - cls_data['median'].values[0])
                    else:
                        values.append(0)
                        errors_low.append(0)
                        errors_high.append(0)
                
                # Calculate bar positions
                n_classes = df_subset.class_abbrev.nunique()
                x_pos = np.arange(n_classes) + i * bar_width - (n_models - 1) * bar_width / 2
                
                # Store for later
                bar_info[model]['x'] = x_pos
                bar_info[model]['height'] = np.array(values) + np.array(errors_high)
                
                # Plot bars
                bars = ax.bar(
                    x_pos,
                    values,
                    bar_width,
                    label=model if row_idx == 0 and col_idx == n_datasets - 1 else "",
                    color=palette[model],
                    alpha=0.8,
                    yerr=[errors_low, errors_high],
                    capsize=3,
                    error_kw={'linewidth': 1.5}
                )
                
                # Annotate bars with values (above error bars)
                for bar, val, err_high in zip(bars, values, errors_high):
                    if val > 0:  # Only annotate non-zero bars
                        # Position text just above the top of the error bar
                        height = val + err_high
                        ax.text(
                            bar.get_x() + bar.get_width() / 2,
                            height + 0.02,
                            f"{val:.2f}",
                            ha="center",
                            va="bottom",
                            fontsize=7,
                            color="black",
                        )
            
            # Add p-value annotations
            pval_subset = pvalues[
                (pvalues['dataset'] == dataset) & 
                (pvalues['metric'] == metric) &
                (pvalues['p_value'] < p_threshold)
            ]
            
            # Find the maximum bar height across all classes to set consistent baseline
            max_bar_height = 0
            for model in models:
                max_bar_height = max(max_bar_height, np.max(bar_info[model]['height']))
            
            for cls_idx, cls in enumerate(dataset_classes):
                pval_cls = pval_subset[pval_subset['class_abbrev'] == cls]
                
                if len(pval_cls) == 0:
                    continue
                
                # Determine which comparisons to show
                if not show_all_comparisons:
                    # Only show comparisons involving the first model (baseline)
                    baseline_model = models[0]
                    comparisons_to_show = pval_cls[
                        (pval_cls['model1_abbrev'] == baseline_model) | 
                        (pval_cls['model2_abbrev'] == baseline_model)
                    ]
                else:
                    # Show all comparisons
                    comparisons_to_show = pval_cls
                
                # Sort by p-value to show most significant first
                comparisons_to_show = comparisons_to_show.sort_values('p_value')
                
                # Consistent vertical spacing parameters
                y_offset = 0.15  # Start height for brackets above tallest bar
                y_step = 0.12    # Vertical spacing between brackets
                
                for comp_idx, (_, row_pval) in enumerate(comparisons_to_show.iterrows()):
                    model1 = row_pval['model1_abbrev']
                    model2 = row_pval['model2_abbrev']
                    p_val = row_pval['p_value']
                    
                    # Get model indices
                    if model1 not in models or model2 not in models:
                        continue
                    
                    # Get x positions
                    x1 = bar_info[model1]['x'][cls_idx]
                    x2 = bar_info[model2]['x'][cls_idx]
                    
                    # Calculate bracket position using consistent baseline
                    y_bracket = max_bar_height + y_offset + comp_idx * y_step
                    
                    # Format p-value as stars
                    p_text = get_significance_marker(p_val)
                    # Alternative: use actual p-values
                    # p_text = format_pvalue(p_val)
                    
                    # Draw bracket
                    ax.plot([x1, x1, x2, x2], 
                           [y_bracket - 0.01, y_bracket, y_bracket, y_bracket - 0.01],
                           'k-', linewidth=1)
                    
                    # Add p-value text
                    ax.text((x1 + x2) / 2, y_bracket + 0.005,
                           p_text,
                           ha='center', va='bottom', fontsize=7, fontweight='bold')
            
            # Formatting
            ax.set_ylabel(metric_names[metric], fontsize=10)
            
            # Title: dataset name on top row
            if row_idx == 0:
                ax.set_title(f"{dataset}", fontsize=11)
            
            # Use fixed-order class labels
            ax.set_xticks(np.arange(len(dataset_classes)))
            ax.set_xticklabels(dataset_classes, rotation=45, ha='right')
            
            # Adjust y-limit to accommodate brackets
            max_y = 1.2
            if len(pval_subset[pval_subset['class_abbrev'].isin(dataset_classes)]) > 0:
                # Count maximum number of brackets for any class in this panel
                max_brackets = 0
                for cls in dataset_classes:
                    pval_cls = pval_subset[pval_subset['class_abbrev'] == cls]
                    if not show_all_comparisons:
                        baseline_model = models[0]
                        n_brackets = len(pval_cls[
                            (pval_cls['model1_abbrev'] == baseline_model) | 
                            (pval_cls['model2_abbrev'] == baseline_model)
                        ])
                    else:
                        n_brackets = len(pval_cls)
                    max_brackets = max(max_brackets, n_brackets)
                
                max_y = 1.2 + max_brackets * y_step
            ax.set_ylim(0, max_y)
            
            # Set y-ticks to only show values up to 1.0
            yticks = ax.get_yticks()
            yticks_filtered = yticks[yticks <= 1.0]
            ax.set_yticks(yticks_filtered)
            ax.set_yticklabels([f'{y:.1f}' for y in yticks_filtered])
            
            # Only show horizontal grid lines
            ax.grid(True, alpha=0.3, axis='y')
            ax.grid(False, axis='x')
            
            # Remove spines
            sns.despine(ax=ax, left=True, bottom=True, right=True, top=True)
            # Re-add spines only for NACC panels
            if dataset == "NACC":
                for spine in ["left", "bottom", "right", "top"]:
                    ax.spines[spine].set_visible(True)
                    ax.spines[spine].set_linewidth(1.0)
    
    # Add legend
    handles = [
        plt.Rectangle((0, 0), 1, 1, fc=palette[m], alpha=0.8, label=m)
        for m in models
    ]
    fig.legend(handles=handles, title="Model", loc="lower center", ncol=3, 
               bbox_to_anchor=(0.15, -0.053), frameon=True)
    
    plt.tight_layout(rect=[0, 0.02, 1, 1])
    
    # Save figure
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, "classwise_precision_recall_with_pvalues.pdf")
    plt.savefig(filename, bbox_inches="tight", dpi=300)
    print(f"Saved {filename}")
    plt.close()


# Example usage:
"""
# Map model names to abbreviations
model_map = {
    "Qwen2.5-3B-Instruct": "Q3B",
    "NACC-3B-OS-SCE": "LUNAR",
    "Qwen2.5-7B-Instruct": "Q7B",
}

# Map class names to abbreviations
class_map = {
    'Mild Cognitive Impairment (MCI)': 'MCI',
    'Cognitively Impaired (IMP)': 'IMP',
    'Dementia (DE)': 'DE',
    'Normal Cognition (NC)': 'NC'
}

# Create the plot with p-value annotations
plot_classwise_with_pvalues(
    all_metrics=all_metrics,
    pairwise_pvalues=pairwise_pvalues,
    model_map=model_map,
    class_map=class_map,
    output_dir="../figure2_classwise",
    show_all_comparisons=False,  # Only show comparisons with baseline (Q3B)
    p_threshold=0.05  # Only show significant results
)
"""

'\n# Map model names to abbreviations\nmodel_map = {\n    "Qwen2.5-3B-Instruct": "Q3B",\n    "NACC-3B-OS-SCE": "LUNAR",\n    "Qwen2.5-7B-Instruct": "Q7B",\n}\n\n# Map class names to abbreviations\nclass_map = {\n    \'Mild Cognitive Impairment (MCI)\': \'MCI\',\n    \'Cognitively Impaired (IMP)\': \'IMP\',\n    \'Dementia (DE)\': \'DE\',\n    \'Normal Cognition (NC)\': \'NC\'\n}\n\n# Create the plot with p-value annotations\nplot_classwise_with_pvalues(\n    all_metrics=all_metrics,\n    pairwise_pvalues=pairwise_pvalues,\n    model_map=model_map,\n    class_map=class_map,\n    output_dir="../figure2_classwise",\n    show_all_comparisons=False,  # Only show comparisons with baseline (Q3B)\n    p_threshold=0.05  # Only show significant results\n)\n'

In [95]:

# Example usage:
# Map model names to abbreviations
model_map = {
    "Qwen2.5-3B-Instruct": "Q3B",
    "NACC-3B-OS-SCE": "LUNAR",
    "Qwen2.5-7B-Instruct": "Q7B",
}

# Map class names to abbreviations
class_map = {
    'Mild Cognitive Impairment (MCI)': 'MCI',
    # 'Cognitively Impaired (IMP)': 'IMP',
    'Dementia (DE)': 'DE',
    'Normal Cognition (NC)': 'NC'
}

# Create the plot with p-value annotations
plot_classwise_with_pvalues(
    all_metrics=all_metrics,
    pairwise_pvalues=pairwise_pvalues,
    model_map=model_map,
    class_map=class_map,
    output_dir="../figure2_classwise",
    show_all_comparisons=True,  # Only show comparisons with baseline (Q3B)
    p_threshold=1  # Only show significant results
)

Saved ../figure2_classwise/classwise_precision_recall_with_pvalues.pdf


In [98]:
# pairwise_pvalues.to_csv('../figure2_classwise/pvalues.csv')